In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from pymongo import MongoClient


def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

client = MongoClient("mongodb://database:27017/")
db = client["proyecto_bigdata"]
coleccion = db["datos_scraping"]

URL = "https://seguimientoejecucionobras.coordinador.cl/"

driver = iniciar_navegador()
datos = []

try:
    print("Conectando...")
    driver.get(URL)

    time.sleep(10) 
    
    tablas = driver.find_elements(By.TAG_NAME, "table")
    print(f"Tablas encontradas: {len(tablas)}")

    for tabla in tablas:
        filas = tabla.find_elements(By.TAG_NAME, "tr")

        for fila in filas:
            columnas = fila.find_elements(By.TAG_NAME, "td")

            if len(columnas) >= 2:
                nombre = columnas[0].text.strip()
                valor = columnas[1].text.strip()

                if nombre and valor:
                    documento = {
                        "campo": nombre,
                        "valor": valor,
                        "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                        "grupo": "G1_Energia"
                    }

                    datos.append(documento)

                    coleccion.insert_one(documento)

    if datos:
        df = pd.DataFrame(datos)
        archivo = "trazabilidad_energia.csv"
        df.to_csv(archivo, index=False)

        print(f"Datos guardados en: {archivo}")
        print(df.head())
        print("Datos guardados en MongoDB")

    else:
        print("No se capturaron datos")

except Exception as e:
    print(f"Error: {e}")

finally:
    driver.quit()
    print("Navegador cerrado")

Conectando...
